# Диагностика C40

Тематический этап подготовки Европы 2019–2023. Выполняйте ячейки сверху вниз в окружении проекта. Каждый ноутбук соответствует одному шагу; диагностика и применение согласованного изменения разделены. Входы, выходы и порядок описаны в [карте подготовки](README.md). Сохраненные результаты перенесены из выполненного общего ноутбука; сам перенос данных не преобразует.

## Шаг 7 – Временная диагностика C40

Проверяем все 7 984 254 записи рабочей копии после справочного обогащения. C40 – метаданные входа в ASMA, 40-мильную область вокруг опорной точки аэропорта; это не отметка входа на глиссаду ILS.

Первичный источник полей – [описание автора на Zenodo](https://zenodo.org/records/10210178). [Статья авторов, разделы 2.2–2.3](https://doi.org/10.59490/joas.2023.7219) объясняет разные источники: посадочные метаданные из OpenSky и ASMA из EUROCONTROL; сопоставление по дню, аэропорту и ICAO24. Эти ключи не гарантируют однозначности конкретной записи.

[EUROCONTROL/EASA AVENIR, раздел 5.4, PDF-страница 43](https://www.eurocontrol.int/sites/default/files/2025-09/avenir-wg-pillar-1-final-report.pdf) описывает переход от последнего пересечения 40 NM к первому входу в обновленной методике. Это допускает гипотезу повторного входа после ухода, но не устанавливает алгоритм формирования нашего файла. Публичный ноутбук автора использует готовую таблицу; код ее исходного сопоставления в нем не найден.

Отдельно проверяем C40 после первой попытки, после последней попытки и после посадки. Группы пересекаются: 42 записи входят в группу из 87, их нельзя складывать. Проверяем пропуски и первый элемент списка попыток относительно его минимума. Соседние записи того же адреса в окне ±24 часа служат только диагностическими кандидатами, а не правилом исправления. Все времена в UTC; допуск и временные сдвиги не применяются.

In [1]:
# Проверяем положение времени C40 относительно попыток и посадки
from pathlib import Path
import runpy
import json
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
# Диагностика сканирует всю базу и сохраняет категории, строки и пары для разбора
module = runpy.run_path(str(ROOT / 'europe/scripts/diagnose_c40.py'))
c40_flags, c40_coverage, c40_records, c40_pairs = module['main']()
c40_journal = json.loads((ROOT / 'europe/reports/preparation/06_c40_diagnostics.json').read_text(encoding='utf8'))
# Показываем счетчики и подтверждение неизменности входных файлов
display(c40_flags)
display(pd.DataFrame(c40_journal['categories']))
print('Уникальных записей для разбора:', c40_journal['unique_review_rows'])
print('Исходник и рабочая копия неизменны:', c40_journal['source_unchanged'], c40_journal['input_unchanged'])

Проверено строк: 1000000


Проверено строк: 2000000


Проверено строк: 3000000


Проверено строк: 4000000


Проверено строк: 5000000


...

Проверено строк: 7000000


{
  "checked_date": "2026-10-08",
  "input": "europe\\data\\working\\02_aircraft_reference_enriched.parquet",
  "input_sha256": "25b6010d4d7197a22a41ee597fd3675bd325ae43107af6709d7c33f96f097187",
  "input_unchanged": true,
  "source_sha256": "1160bdbea6fa93249a00768452be64047c6d2461bb0843d5ecfc52e6942c7f02",
  "source_unchanged": true,
  "rows_checked": 7984254,
  "go_arounds": 26253,
  "rows_changed": 0,
  "rows_deleted": 0,
  "flags": [
    {
      "flag": "after_first_attempt",
      "rows": 87,
      "go_arounds": 45
    },
    {
      "flag": "after_landing",
      "rows": 42,
      "go_arounds": 0
    },
    {
      "flag": "after_last_attempt",
      "rows": 42,
      "go_arounds": 0
    },
    {
      "flag": "equal_first_attempt",
      "rows": 0,
      "go_arounds": 0
    },
    {
      "flag": "equal_landing",
      "rows": 0,
      "go_arounds": 0
    }
  ],
  "unique_review_rows": 87,
  "first_attempt_not_earliest_rows": 0,
  "null_attempt_lists": 0,
  "null_attempt_elemen

,flag,rows,go_arounds
0,after_first_attempt,87,45
1,after_landing,42,0
2,after_last_attempt,42,0
3,equal_first_attempt,0,0
4,equal_landing,0,0


,category,rows,go_arounds
0,after_first_before_or_equal_landing,45,45
1,after_landing,42,0
2,before_first_attempt,6331785,18361
3,missing_c40,1652382,7847


Уникальных записей для разбора: 87
Исходник и рабочая копия неизменны: True True


In [2]:
# Сводим записи по аэропортам и временным категориям C40
print('Распределение 87 записей по аэропортам и группам:')
display(c40_records.groupby(['airport', 'c40_time_category']).agg(
    rows=('GoA', 'size'), go_arounds=('GoA', 'sum'),
    min_landing_minus_c40_minutes=('landing_minus_c40_minutes', 'min'),
    max_landing_minus_c40_minutes=('landing_minus_c40_minutes', 'max')).reset_index())

# Выводим все случаи времени C40 после посадки с ключевыми временами
print('Все 42 записи с C40 после посадки:')
c40_review_columns = ['source_row_position', 'flight_id', 'icao24', 'airport', 'landing_time',
                     'first_attempt_time', 'last_attempt_time', 'C40_CROSS_TIME',
                     'GoA', 'n_attempts', 'landing_minus_c40_minutes']
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(c40_records.loc[c40_records['after_landing'], c40_review_columns])

Распределение 87 записей по аэропортам и группам:


,airport,c40_time_category,rows,go_arounds,min_landing_minus_c40_minutes,max_landing_minus_c40_minutes
0,EDDB,after_first_before_or_equal_landing,1,1,23.000000,23.000000
1,EDDF,after_first_before_or_equal_landing,6,6,12.733333,30.883333
2,EDDM,after_first_before_or_equal_landing,2,2,11.200000,14.100000
3,EGLL,after_landing,2,0,-5.133333,-0.433333
4,EGSS,after_landing,1,0,-2.333333,-2.333333
5,EHAM,after_first_before_or_equal_landing,2,2,17.533333,21.966667
6,EHAM,after_landing,1,0,-1.633333,-1.633333
7,EKCH,after_landing,1,0,-6.616667,-6.616667
8,LEMD,after_first_before_or_equal_landing,16,16,11.550000,27.083333
9,LEMD,after_landing,3,0,-9.200000,-2.433333


Все 42 записи с C40 после посадки:


,source_row_position,flight_id,icao24,airport,landing_time,first_attempt_time,last_attempt_time,C40_CROSS_TIME,GoA,n_attempts,landing_minus_c40_minutes
0,696773,VLG6273_23112,3451d3,LIRF,2019-04-19 22:05:09+00:00,2019-04-19 22:05:09+00:00,2019-04-19 22:05:09+00:00,2019-04-19 22:12:02+00:00,False,1,-6.883333
1,916854,VLG80EU_15642,34538d,LFPO,2019-05-21 20:30:18+00:00,2019-05-21 20:30:18+00:00,2019-05-21 20:30:18+00:00,2019-05-21 20:34:02+00:00,False,1,-3.733333
2,1223930,IBS37HF_10828,341682,LEMD,2019-07-02 20:52:51+00:00,2019-07-02 20:50:22+00:00,2019-07-02 20:50:22+00:00,2019-07-02 21:02:03+00:00,False,1,-9.200000
3,1474355,EXS1458_14252,4070ea,EGSS,2019-08-05 13:05:45+00:00,2019-08-05 13:05:45+00:00,2019-08-05 13:05:45+00:00,2019-08-05 13:08:05+00:00,False,1,-2.333333
4,1477978,ABR6EC_178801,4ca282,LFPG,2019-08-05 22:06:17+00:00,2019-08-05 22:04:27+00:00,2019-08-05 22:04:27+00:00,2019-08-05 22:14:18+00:00,False,1,-8.016667
5,1722891,ANE15HD_99479,345543,LEMD,2019-09-07 08:26:24+00:00,2019-09-07 08:25:59+00:00,2019-09-07 08:25:59+00:00,2019-09-07 08:28:50+00:00,False,1,-2.433333
6,2004968,EZS65DW_62153,4b1a65,LIRF,2019-10-15 18:32:18+00:00,2019-10-15 18:30:09+00:00,2019-10-15 18:30:09+00:00,2019-10-15 18:40:50+00:00,False,1,-8.533333
7,2598399,DAH1144_4589,0a0077,LFPG,2020-01-13 14:49:56+00:00,2020-01-13 14:49:56+00:00,2020-01-13 14:49:56+00:00,2020-01-13 14:50:44+00:00,False,1,-0.800000
8,2600997,ABR6BN_82206,4ca282,LFPG,2020-01-13 22:53:47+00:00,2020-01-13 22:53:28+00:00,2020-01-13 22:53:28+00:00,2020-01-13 22:56:21+00:00,False,1,-2.566667
10,2766525,EZS52QB_28471,4b1a2a,LSGG,2020-02-08 20:45:50+00:00,2020-02-08 20:45:46+00:00,2020-02-08 20:45:46+00:00,2020-02-08 20:50:30+00:00,False,1,-4.666667


In [3]:
# Выделяем C40 после первой попытки, исключая случаи после посадки
print('Все 45 записей с C40 после первой попытки, но до посадки:')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(c40_records.loc[c40_records['after_first_attempt'] & ~c40_records['after_landing'], c40_review_columns])
# Проверяем повторное использование одной отметки у того же борта и аэропорта
print('Повторное присоединение той же отметки C40 к другому наблюдению того же адреса и аэропорта:')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(c40_pairs.loc[c40_pairs['same_airport'] & c40_pairs['same_c40_time']])
print('Это свидетельство неоднозначности сопоставления; исправления и объединение записей не выполнены.')

Все 45 записей с C40 после первой попытки, но до посадки:


,source_row_position,flight_id,icao24,airport,landing_time,first_attempt_time,last_attempt_time,C40_CROSS_TIME,GoA,n_attempts,landing_minus_c40_minutes
9,2734305,TVF4511_19553,3964f5,LFPO,2020-02-03 21:40:50+00:00,2020-02-03 21:03:23+00:00,2020-02-03 21:40:37+00:00,2020-02-03 21:20:13+00:00,True,2,20.616667
11,2771720,OMA131_87474,70c0ef,LFPG,2020-02-09 19:08:56+00:00,2020-02-09 18:26:40+00:00,2020-02-09 19:08:56+00:00,2020-02-09 18:51:25+00:00,True,2,17.516667
12,2771854,TRA4H_64916,4853d2,EHAM,2020-02-09 19:30:15+00:00,2020-02-09 18:44:34+00:00,2020-02-09 19:30:15+00:00,2020-02-09 19:08:17+00:00,True,2,21.966667
14,3000179,TVF93PC_49784,3964f7,LEMD,2020-03-15 18:52:59+00:00,2020-03-15 18:24:15+00:00,2020-03-15 18:52:59+00:00,2020-03-15 18:39:59+00:00,True,2,13.000000
17,3237730,DLH1UP_49736,3c65d0,EDDF,2020-08-13 06:38:03+00:00,2020-08-13 05:57:23+00:00,2020-08-13 06:37:44+00:00,2020-08-13 06:07:10+00:00,True,3,30.883333
21,3519971,SWT2KT_6372,344485,LFPG,2020-11-11 10:24:28+00:00,2020-11-11 08:19:41+00:00,2020-11-11 10:24:28+00:00,2020-11-11 09:04:30+00:00,True,2,79.966667
23,3615225,SXS5FC_87009,4bcde4,EHAM,2020-12-27 07:54:45+00:00,2020-12-27 06:54:23+00:00,2020-12-27 07:54:25+00:00,2020-12-27 07:37:13+00:00,True,3,17.533333
24,3615785,AUI3FU_86756,508371,LFPG,2020-12-27 11:58:28+00:00,2020-12-27 11:04:59+00:00,2020-12-27 11:58:23+00:00,2020-12-27 11:41:43+00:00,True,2,16.750000
25,3762911,UAE9993_120041,89616f,EDDF,2021-03-13 11:05:33+00:00,2021-03-13 10:37:06+00:00,2021-03-13 11:05:12+00:00,2021-03-13 10:47:19+00:00,True,2,18.233333
26,3904680,DLH46J_24768,3c5462,EDDF,2021-05-16 18:14:07+00:00,2021-05-16 17:20:27+00:00,2021-05-16 18:13:40+00:00,2021-05-16 17:46:07+00:00,True,2,28.000000


Повторное присоединение той же отметки C40 к другому наблюдению того же адреса и аэропорта:


,flagged_row_position,related_row_position,icao24,flagged_airport,related_airport,same_airport,same_c40_time,same_c40_lat_lon,same_callsign,related_landing_minus_c40_minutes,landing_difference_minutes
17,1223930,1224005,341682,LEMD,LEMD,True,True,True,True,10.233333,19.433333
23,1477978,1478031,4ca282,LFPG,LFPG,True,True,True,True,11.466667,19.483333
33,2004968,2005110,4b1a65,LIRF,LIRF,True,True,True,True,11.266667,19.800000


Это свидетельство неоднозначности сопоставления; исправления и объединение записей не выполнены.


## Итог шага 7 – Правило очистки пока не применено

Полный проход подтвердил аудит 01: 87 записей имеют C40 после первой попытки. В их составе 45 отмеченных уходов с 2–3 попытками: C40 предшествует последней попытке и посадке. Такие случаи совместимы с повторным входом, но требуют траекторий для подтверждения. Эта отметка C40 не подходит как момент прогноза до первой попытки; переносить ее назад искусственно нельзя.

Другие 42 записи имеют одну попытку, GoA=False и C40 через 26 секунд – 9 минут 12 секунд после указанной посадки. Это внутреннее противоречие метаданных. В 23 случаях аэропорт – LSGG; концентрация не является основанием исключить весь аэропорт. Три записи имеют тот же C40 и координаты, что более позднее наблюдение того же адреса, аэропорта и позывного. Это может отражать сопоставление или сегментацию; определить ошибочное поле и исправить метку ухода по одной таблице нельзя.

C40 отсутствует в 1 652 382 записях, включая 7 847 уходов. Пропуски не приравниваем к противоречиям. В LTFJ C40 отсутствует во всех записях; статья объясняет отсутствие покрытия в EU-NM. Для остальных аэропортов не переносим проценты покрытия статьи на наш полный файл.

Предлагаем следующим согласованным действием добавить только диагностические флаги в отдельную рабочую копию: C40 отсутствует; C40 после первой попытки; C40 после посадки. Все записи и исходные поля сохранить. Для будущего анализа отдельно определить допустимость C40 относительно момента прогноза. Обнуление C40, исключение строк, изменение GoA, объединение сегментов и исключение аэропортов сейчас не выполняются.

Временные диагностические флаги используют сведения завершенного наблюдения и не предназначены для подачи в модель раннего прогноза. Отбор по ним может изменить состав целевых классов и требует отдельной оценки.

Подробный [разбор C40](../../reports/preparation/06_c40_review.md) содержит ограничения и источники. Проверены SHA256 исходника и рабочей копии до и после проходов; изменений нет.